# Nanochat Distill Run on Kaggle

This notebook runs a Kaggle-friendly distillation flow for the `d8_kaggle` chat model:

- Uses `meta-llama/Llama-3.1-8B-Instruct` as teacher.
- Imports the saved SFT checkpoint cache from `nanochat-d8-sft-cache`.
- Generates teacher data in restartable JSONL shards.
- Keeps Hugging Face downloads in `/kaggle/temp` to protect `/kaggle/working` output space.
- Trains a distilled student from `sft:d8_kaggle` and writes `chatdistill_checkpoints/d8_kaggle`.

Recommended Kaggle setup: GPU on, Internet on, attach `nanochat-codes` and `nanochat-d8-sft-cache`.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

INPUT_ROOT = Path("/kaggle/input")

CODE_INPUTS = sorted(INPUT_ROOT.glob("**/nanochat-codes"))
SFT_CACHE_INPUTS = sorted(INPUT_ROOT.glob("**/nanochat-d8-sft-cache"))

assert CODE_INPUTS, "Attach the nanochat-codes Kaggle dataset"
assert SFT_CACHE_INPUTS, "Attach the nanochat-d8-sft-cache Kaggle dataset"

CODE_INPUT = CODE_INPUTS[0]
SFT_CACHE_INPUT = SFT_CACHE_INPUTS[0]

WORK_REPO = Path("/kaggle/working/nanochat")
WORK_CACHE = Path("/kaggle/working/nanochat_cache")
HF_CACHE = Path("/kaggle/temp/huggingface_cache")

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

# Distillation from sft:d8_kaggle needs the tokenizer and SFT checkpoint.
required_cache_paths = [
    Path("tokenizer"),
    Path("chatsft_checkpoints") / "d8_kaggle",
]
for rel_path in required_cache_paths:
    src = SFT_CACHE_INPUT / rel_path
    dst = WORK_CACHE / rel_path
    assert src.exists(), f"Missing required cache path in attached dataset: {src}"
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print("Code input:", CODE_INPUT)
print("SFT cache input:", SFT_CACHE_INPUT)
print("Working repo:", WORK_REPO)
print("Working cache:", WORK_CACHE)
print("HF cache:", HF_CACHE)
subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/temp"], check=False)
subprocess.run(["du", "-sh", str(WORK_CACHE)], check=False)


Code input: /kaggle/input/datasets/jennyqqjiang/nanochat-codes
SFT cache input: /kaggle/input/datasets/yshuaiqin/nanochat-d8-sft-cache
Working repo: /kaggle/working/nanochat
Working cache: /kaggle/working/nanochat_cache
HF cache: /kaggle/temp/huggingface_cache
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.1G   18G  11% /kaggle/working
overlay         7.9T  6.8T  1.1T  87% /
2.1G	/kaggle/working/nanochat_cache


CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache'], returncode=0)

## Install Dependencies

This run needs `bitsandbytes` for 8-bit Llama loading. The `only-if-needed` strategy avoids unnecessary upgrades of Kaggle's preinstalled CUDA/RAPIDS packages.

In [2]:
packages = [
    "accelerate>=1.0.0",
    "bitsandbytes>=0.46.1",
    "datasets>=4.0.0",
    "filelock>=3.0.0",
    "psutil>=7.1.0",
    "requests>=2.32.0",
    "safetensors>=0.4.0",
    "sentencepiece>=0.2.0",
    "tiktoken>=0.11.0",
    "tokenizers>=0.22.0",
    "transformers>=4.57.3",
    "wandb>=0.21.3",
    "zstandard>=0.25.0",
    "rustbpe>=0.1.0",
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade-strategy",
    "only-if-needed",
] + packages)
print("Dependencies installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.6 MB/s eta 0:00:00
Dependencies installed


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

## Configure Runtime

In [3]:
env = os.environ.copy()
env["NANOCHAT_BASE_DIR"] = str(WORK_CACHE)
env["HF_HOME"] = str(HF_CACHE)
env["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
env["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
env["HF_HUB_DISABLE_XET"] = "1"
env["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
env["HF_HUB_ETAG_TIMEOUT"] = "60"
env["TMPDIR"] = "/kaggle/temp"
env["PYTHONUNBUFFERED"] = "1"

# chat_distill.py does not use GradScaler, so fp32 is safer on T4 for student training.
env["NANOCHAT_DTYPE"] = "float32"

env["NANOCHAT_DISABLE_COMPILE"] = "1"
env["TORCH_COMPILE_DISABLE"] = "1"
env["NANOCHAT_ADAMW_ALLREDUCE"] = "1"
env["NANOCHAT_SERIAL_OPTIM_COMM"] = "1"
env["OMP_NUM_THREADS"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"
env["TORCH_NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_DEBUG"] = "WARN"

os.environ.update(env)

print("NANOCHAT_BASE_DIR:", env["NANOCHAT_BASE_DIR"])
print("HF_HOME:", env["HF_HOME"])
print("HF_HUB_DISABLE_XET:", env["HF_HUB_DISABLE_XET"])
print("NANOCHAT_DTYPE:", env["NANOCHAT_DTYPE"])


NANOCHAT_BASE_DIR: /kaggle/working/nanochat_cache
HF_HOME: /kaggle/temp/huggingface_cache
HF_HUB_DISABLE_XET: 1
NANOCHAT_DTYPE: float32


In [4]:
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
CUDA device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## Hugging Face Token

In [ ]:
import getpass

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"]:
    os.environ["HF_TOKEN"] = "" #getpass.getpass("HF_TOKEN: ") your HF token
env["HF_TOKEN"] = os.environ["HF_TOKEN"]
print("HF_TOKEN configured")


HF_TOKEN configured


## Validate Repo And Cache

In [6]:
assert (WORK_REPO / "scripts" / "chat_distill_data.py").exists(), "Missing scripts/chat_distill_data.py"
assert (WORK_REPO / "scripts" / "chat_distill.py").exists(), "Missing scripts/chat_distill.py"
assert (WORK_REPO / "nanochat" / "checkpoint_manager.py").exists(), "Missing nanochat/checkpoint_manager.py"

sft_dir = WORK_CACHE / "chatsft_checkpoints" / "d8_kaggle"
tokenizer_dir = WORK_CACHE / "tokenizer"
assert sft_dir.exists(), f"Missing SFT checkpoint dir: {sft_dir}"
assert tokenizer_dir.exists(), f"Missing tokenizer dir: {tokenizer_dir}"

subprocess.check_call(
    [
        sys.executable, "-m", "py_compile",
        "scripts/chat_distill_data.py",
        "scripts/chat_distill.py",
        "nanochat/checkpoint_manager.py",
    ],
    cwd=WORK_REPO,
    env=env,
)

print("Setup complete")
print("SFT checkpoints:", sorted(p.name for p in sft_dir.glob("model_*.pt"))[-5:])
print("Tokenizer files:", sorted(p.name for p in tokenizer_dir.iterdir()))


Setup complete
SFT checkpoints: ['model_004999.pt']
Tokenizer files: ['token_bytes.pt', 'tokenizer.pkl']


## Distill Run Settings

Defaults here are intentionally small enough for a Kaggle pass. If teacher generation is too slow, reduce `TOTAL_EXAMPLES` or `CHUNK_SIZE`.


In [7]:
TEACHER_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

TOTAL_EXAMPLES = 128
CHUNK_SIZE = 32
START_EXAMPLE = 0
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.0

DISTILL_FORMAT = "sft"
DISTILL_DATA = WORK_CACHE / "teacher_sft_distill.jsonl"
SHARD_DIR = WORK_CACHE / "teacher_sft_distill_shards"
SHARD_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    "You are a precise math tutor. Solve carefully and concisely. "
    "Always end with exactly one final line formatted as: #### <answer>"
)

print("Teacher:", TEACHER_MODEL)
print("Examples:", TOTAL_EXAMPLES)
print("Chunk size:", CHUNK_SIZE)
print("Max new tokens:", MAX_NEW_TOKENS)
print("Output:", DISTILL_DATA)


Teacher: meta-llama/Llama-3.1-8B-Instruct
Examples: 128
Chunk size: 32
Max new tokens: 128
Output: /kaggle/working/nanochat_cache/teacher_sft_distill.jsonl


## Generate Teacher SFT Shards

Each shard is an independent JSONL file. If the notebook disconnects, rerun this cell and completed shards will be skipped.

In [8]:
def count_jsonl(path):
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

for start in range(START_EXAMPLE, START_EXAMPLE + TOTAL_EXAMPLES, CHUNK_SIZE):
    end = min(START_EXAMPLE + TOTAL_EXAMPLES, start + CHUNK_SIZE)
    expected = end - start
    shard_path = SHARD_DIR / f"teacher_sft_{start:06d}_{end:06d}.jsonl"
    existing = count_jsonl(shard_path)
    if existing == expected:
        print(f"Skipping complete shard {shard_path.name} ({existing} rows)")
        continue
    if shard_path.exists():
        print(f"Regenerating incomplete shard {shard_path.name}: {existing}/{expected} rows")

    cmd = [
        sys.executable,
        "-m", "scripts.chat_distill_data",
        "--teacher-model", TEACHER_MODEL,
        "--input-source", "gsm8k",
        "--split", "train",
        "--start", str(start),
        "--max-examples", str(expected),
        "--output-format", DISTILL_FORMAT,
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--temperature", str(TEMPERATURE),
        "--top-p", "0.95",
        "--torch-dtype", "float16",
        "--device-map", "auto",
        "--load-in-8bit", "1",
        "--chat-style", "solution",
        "--system-prompt", SYSTEM_PROMPT,
        "--output-path", str(shard_path),
    ]

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)
    got = count_jsonl(shard_path)
    assert got == expected, f"Shard {shard_path} has {got} rows, expected {expected}"
    subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/temp"], check=False)



Running: /usr/bin/python3 -m scripts.chat_distill_data --teacher-model meta-llama/Llama-3.1-8B-Instruct --input-source gsm8k --split train --start 0 --max-examples 32 --output-format sft --max-new-tokens 128 --temperature 0.0 --top-p 0.95 --torch-dtype float16 --device-map auto --load-in-8bit 1 --chat-style solution --system-prompt You are a precise math tutor. Solve carefully and concisely. Always end with exactly one final line formatted as: #### <answer> --output-path /kaggle/working/nanochat_cache/teacher_sft_distill_shards/teacher_sft_000000_000032.jsonl


Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 259743.98 examples/s]


Loaded 32 prompt examples from gsm8k


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:23<00:00, 12.50it/s, Materializing param=model.norm.weight]


Loaded teacher model: meta-llama/Llama-3.1-8B-Instruct


...

Wrote 10/32 examples to /kaggle/working/nanochat_cache/teacher_sft_distill_shards/teacher_sft_000064_000096.jsonl
Wrote 20/32 examples to /kaggle/working/nanochat_cache/teacher_sft_distill_shards/teacher_sft_000064_000096.jsonl
Wrote 30/32 examples to /kaggle/working/nanochat_cache/teacher_sft_distill_shards/teacher_sft_000064_000096.jsonl
Wrote 32/32 examples to /kaggle/working/nanochat_cache/teacher_sft_distill_shards/teacher_sft_000064_000096.jsonl
Finished writing teacher data to /kaggle/working/nanochat_cache/teacher_sft_distill_shards/teacher_sft_000064_000096.jsonl
Running: /usr/bin/python3 -m scripts.chat_distill_data --teacher-model meta-llama/Llama-3.1-8B-Instruct --input-source gsm8k --split train --start 96 --max-examples 32 --output-format sft --max-new-tokens 128 --temperature 0.0 --top-p 0.95 --torch-dtype float16 --device-map auto --load-in-8bit 1 --chat-style solution --system-prompt You are a precise math tutor. Solve carefully and concisely. Always end with exactly o

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:21<00:00, 13.45it/s, Materializing param=model.norm.weight]


Loaded teacher model: meta-llama/Llama-3.1-8B-Instruct


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Wrote 10/32 examples to /kaggle/working/nanochat_cache/teacher_sft_distill_shards/teacher_sft_000096_000128.jsonl
Wrote 20/32 examples to /kaggle/working/nanochat_cache/teacher_sft_distill_shards/teacher_sft_000096_000128.jsonl
Wrote 30/32 examples to /kaggle/working/nanochat_cache/teacher_sft_distill_shards/teacher_sft_000096_000128.jsonl
Wrote 32/32 examples to /kaggle/working/nanochat_cache/teacher_sft_distill_shards/teacher_sft_000096_000128.jsonl
Finished writing teacher data to /kaggle/working/nanochat_cache/teacher_sft_distill_shards/teacher_sft_000096_000128.jsonl
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.1G   18G  11% /kaggle/working
overlay         7.9T  6.9T  1.1T  87% /


## Merge And Inspect Teacher Data

In [9]:
shard_paths = []
for start in range(START_EXAMPLE, START_EXAMPLE + TOTAL_EXAMPLES, CHUNK_SIZE):
    end = min(START_EXAMPLE + TOTAL_EXAMPLES, start + CHUNK_SIZE)
    shard_paths.append(SHARD_DIR / f"teacher_sft_{start:06d}_{end:06d}.jsonl")

rows = []
for path in shard_paths:
    assert path.exists(), f"Missing shard: {path}"
    with path.open("r", encoding="utf-8") as f:
        rows.extend(line.rstrip("\n") for line in f if line.strip())

DISTILL_DATA.write_text("\n".join(rows) + "\n", encoding="utf-8")

print("Merged data:", DISTILL_DATA)
print("Rows:", len(rows))
assert len(rows) == TOTAL_EXAMPLES, f"Expected {TOTAL_EXAMPLES} rows, got {len(rows)}"

parsed = [json.loads(line) for line in rows]
hash_count = sum("####" in example[-1]["content"] for example in parsed)
print("Rows containing #### in assistant answer:", hash_count, "/", len(parsed))
print(json.dumps(parsed[0], indent=2, ensure_ascii=False)[:3000])

Merged data: /kaggle/working/nanochat_cache/teacher_sft_distill.jsonl
Rows: 128
Rows containing #### in assistant answer: 52 / 128
[
  {
    "role": "system",
    "content": "You are a precise math tutor. Solve carefully and concisely. Always end with exactly one final line formatted as: #### <answer>"
  },
  {
    "role": "user",
    "content": "Solve the following problem carefully. Show a concise reasoning process and finish with a final answer on a new line in the form '#### answer'.\n\nMimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?"
  },
  {
    "role": "assistant",
    "content": "To find the number of seashells Leigh had, we need to follow the sequence of events.\n\n1. Mimi picked up 2 dozen seashells. Since 1 dozen = 12, 2 dozen = 2 * 12 = 24 seashells.\n2. Kyle found twice as many shells as Mimi, so Kyle found 2 * 24 = 

In [10]:
# Many teacher answers do not contain the expected #### pattern.
# So the distill data does not teach the nanochat GSM8K answer format.
# The model may learn some reasoning text, but the evaluator likely cannot extract the final answer.
# Easiest option: postprocess the teacher JSONL to
# append a #### answer line when missing, using GSM8K's gold answer from the source dataset.

work_repo_str = str(WORK_REPO)
if work_repo_str not in sys.path:
    sys.path.insert(0, work_repo_str)

from tasks.gsm8k import GSM8K, extract_answer

task = GSM8K(subset="main", split="train")

fixed_rows = []
fixed = 0

for i, line in enumerate(DISTILL_DATA.read_text(encoding="utf-8").splitlines()):
    row = json.loads(line)
    assistant = row[-1]["content"]

    if "####" not in assistant:
        source_idx = START_EXAMPLE + i
        answer_parts = task[source_idx]["messages"][-1]["content"]
        answer_text = "".join(
            part.get("text", "") for part in answer_parts if isinstance(part, dict)
        )
        gold_answer = extract_answer(answer_text)
        assert gold_answer is not None, (
            f"Could not extract gold answer for GSM8K source index {source_idx}"
        )
        row[-1]["content"] = assistant.rstrip() + f"\n\n#### {gold_answer}"
        fixed += 1

    fixed_rows.append(json.dumps(row, ensure_ascii=False))

DISTILL_DATA.write_text("\n".join(fixed_rows) + "\n", encoding="utf-8")

print("Rows:", len(fixed_rows))
print("Fixed missing ####:", fixed)

Rows: 128
Fixed missing ####: 76


## Optional Preference Data

This is off by default because the distill run already spends most of its time on teacher generation. Enable it only if you also want a preference dataset for later DPO/PPO/GRPO experiments.


In [11]:
RUN_PREFERENCE_VARIANT = False
PREFERENCE_EXAMPLES = 128
PREFERENCE_PATH = WORK_CACHE / "teacher_prefs_distill.jsonl"

if RUN_PREFERENCE_VARIANT:
    cmd = [
        sys.executable,
        "-m", "scripts.chat_distill_data",
        "--teacher-model", TEACHER_MODEL,
        "--input-source", "gsm8k",
        "--split", "train",
        "--output-format", "preference",
        "--max-examples", str(PREFERENCE_EXAMPLES),
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--temperature", str(TEMPERATURE),
        "--top-p", "0.95",
        "--torch-dtype", "float16",
        "--device-map", "auto",
        "--load-in-8bit", "1",
        "--chat-style", "solution",
        "--system-prompt", SYSTEM_PROMPT,
        "--rejected-style", "perturb",
        "--output-path", str(PREFERENCE_PATH),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)
    print("Preference data:", PREFERENCE_PATH)
else:
    print("Skipping preference variant")


Skipping preference variant


## Free Teacher Model Cache Before Student Training

After teacher data is generated, nanochat training no longer needs the Llama checkpoint. Freeing this space makes Kaggle output handling less fragile.

In [12]:
FREE_TEACHER_CACHE_AFTER_DATA = True

if FREE_TEACHER_CACHE_AFTER_DATA:
    for path in [HF_CACHE / "hub" / "models--meta-llama--Llama-3.1-8B-Instruct", HF_CACHE / "xet"]:
        if path.exists():
            shutil.rmtree(path)
            print("Removed:", path)
else:
    print("Keeping teacher cache")

subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/temp"], check=False)


Removed: /kaggle/temp/huggingface_cache/hub/models--meta-llama--Llama-3.1-8B-Instruct
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.1G   18G  11% /kaggle/working
overlay         7.9T  6.8T  1.1T  87% /


CompletedProcess(args=['df', '-h', '/kaggle/working', '/kaggle/temp'], returncode=0)

## Train Distilled Student With Gentler LRs

In [13]:
NPROC = 2 if torch.cuda.is_available() and torch.cuda.device_count() >= 2 else 1
STUDENT_SOURCE = "sft"
STUDENT_TAG = "d8_kaggle"

distill_args = [
    "-m", "scripts.chat_distill",
    "--",
    "--run", "dummy",
    "--student-source", STUDENT_SOURCE,
    "--student-tag", STUDENT_TAG,
    "--data-path", str(DISTILL_DATA),
    "--data-format", DISTILL_FORMAT,
    "--val-ratio", "0.1",
    "--num-epochs", "1",
    "--batch-size", "2",
    "--max-seq-len", "1024",
    "--embedding-lr", "0.01",
    "--matrix-lr", "0.001",
    "--unembedding-lr", "0.0002",
    "--warmup-ratio", "0.1",
    "--final-lr-frac", "0.1",
    "--eval-every", "25",
    "--save-every", "1000",
]

if NPROC > 1:
    cmd = ["torchrun", "--standalone", f"--nproc_per_node={NPROC}"] + distill_args
else:
    cmd = [sys.executable] + distill_args

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


Running: torchrun --standalone --nproc_per_node=2 -m scripts.chat_distill -- --run dummy --student-source sft --student-tag d8_kaggle --data-path /kaggle/working/nanochat_cache/teacher_sft_distill.jsonl --data-format sft --val-ratio 0.1 --num-epochs 1 --batch-size 2 --max-seq-len 1024 --embedding-lr 0.01 --matrix-lr 0.001 --unembedding-lr 0.0002 --warmup-ratio 0.1 --final-lr-frac 0.1 --eval-every 25 --save-every 1000


[W614 03:24:33.242584456 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


Autodetected device type: cuda


[W614 03:24:39.743734472 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 03:24:39.751386474 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


2026-06-14 03:24:40,021 - nanochat.common - INFO - Distributed world size: 2
2026-06-14 03:24:40,022 - nanochat.checkpoint_manager - INFO - Loading model from /kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle with step 4999
2026-06-14 03:24:42,078 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 1024, 'vocab_size': 32768, 'n_layer': 8, 'n_head': 4, 'n_kv_head': 4, 'n_embd': 512, 'window_pattern': 'L'}


Distill dataset: train=116 | val=12
Scaling the LR for the AdamW parameters ∝1/√(512/768) = 1.224745
Planned distillation steps: 29
Step 0 | val_loss: 1.287748
step 00000 | epoch 0 | loss: 1.471811 | lrm: 0.2000 | dt: 444.48ms
step 00001 | epoch 0 | loss: 1.338727 | lrm: 0.4857 | dt: 44.83ms
step 00002 | epoch 0 | loss: 1.415962 | lrm: 0.7714 | dt: 69.77ms
step 00003 | epoch 0 | loss: 1.376617 | lrm: 0.9929 | dt: 43.77ms
step 00004 | epoch 0 | loss: 1.452479 | lrm: 0.9571 | dt: 42.66ms
step 00005 | epoch 0 | loss: 1.531057 | lrm: 0.9214 | dt: 40.48ms
step 00006 | epoch 0 | loss: 1.606931 | lrm: 0.8857 | dt: 39.92ms
step 00007 | epoch 0 | loss: 1.573282 | lrm: 0.8500 | dt: 37.30ms
step 00008 | epoch 0 | loss: 1.565884 | lrm: 0.8143 | dt: 37.46ms
step 00009 | epoch 0 | loss: 1.483196 | lrm: 0.7786 | dt: 39.55ms
step 00010 | epoch 0 | loss: 1.484071 | lrm: 0.7429 | dt: 38.26ms
step 00011 | epoch 0 | loss: 1.460663 | lrm: 0.7071 | dt: 37.81ms
step 00012 | epoch 0 | loss: 1.473020 | lrm: 0.

2026-06-14 03:24:52,056 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatdistill_checkpoints/d8_kaggle/model_000029.pt
2026-06-14 03:24:52,057 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatdistill_checkpoints/d8_kaggle/meta_000029.json


Saved final distillation checkpoint to /kaggle/working/nanochat_cache/chatdistill_checkpoints/d8_kaggle


CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '-m', 'scripts.chat_distill', '--', '--run', 'dummy', '--student-source', 'sft', '--student-tag', 'd8_kaggle', '--data-path', '/kaggle/working/nanochat_cache/teacher_sft_distill.jsonl', '--data-format', 'sft', '--val-ratio', '0.1', '--num-epochs', '1', '--batch-size', '2', '--max-seq-len', '1024', '--embedding-lr', '0.01', '--matrix-lr', '0.001', '--unembedding-lr', '0.0002', '--warmup-ratio', '0.1', '--final-lr-frac', '0.1', '--eval-every', '25', '--save-every', '1000'], returncode=0)

## Inspect Distill Checkpoints

In [14]:
distill_dir = WORK_CACHE / "chatdistill_checkpoints" / STUDENT_TAG
assert distill_dir.exists(), f"Missing distill checkpoint dir: {distill_dir}"

model_files = sorted(distill_dir.glob("model_*.pt"))
meta_files = sorted(distill_dir.glob("meta_*.json"))

print("Distill dir:", distill_dir)
print("Model checkpoints:", [p.name for p in model_files])
print("Metadata files:", [p.name for p in meta_files])

if meta_files:
    print(json.dumps(json.loads(meta_files[-1].read_text()), indent=2)[:3000])

subprocess.run(["du", "-sh", str(WORK_CACHE)], check=False)


Distill dir: /kaggle/working/nanochat_cache/chatdistill_checkpoints/d8_kaggle
Model checkpoints: ['model_000029.pt']
Metadata files: ['meta_000029.json']
{
  "step": 29,
  "best_val_loss": 1.1101728677749634,
  "model_config": {
    "sequence_len": 1024,
    "vocab_size": 32768,
    "n_layer": 8,
    "n_head": 4,
    "n_kv_head": 4,
    "n_embd": 512,
    "window_pattern": "L"
  },
  "user_config": {
    "run": "dummy",
    "device_type": "",
    "student_source": "sft",
    "student_tag": "d8_kaggle",
    "student_step": null,
    "data_path": "/kaggle/working/nanochat_cache/teacher_sft_distill.jsonl",
    "data_format": "sft",
    "val_ratio": 0.1,
    "shuffle_seed": 42,
    "num_epochs": 1,
    "batch_size": 2,
    "max_seq_len": 1024,
    "embedding_lr": 0.01,
    "unembedding_lr": 0.0002,
    "matrix_lr": 0.001,
    "weight_decay": 0.0,
    "init_lr_frac": 0.2,
    "warmup_ratio": 0.1,
    "final_lr_frac": 0.1,
    "eval_every": 25,
    "save_every": 1000
  }
}
2.6G	/kaggle/worki

CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache'], returncode=0)

## Comparison Eval

Use a larger limit than the smoke test if time allows. Scores are still noisy, but this is enough to catch obvious regressions.

In [15]:
RUN_COMPARISON_EVAL = True
EVAL_MAX_PROBLEMS = 50

if RUN_COMPARISON_EVAL:
    post_eval_args = [
        "-m", "scripts.chat_post_eval",
        "--",
        "--models", f"sft=sft:{STUDENT_TAG}",
        "--models", f"distill=@{WORK_CACHE / 'chatdistill_checkpoints'}:{STUDENT_TAG}",
        "--max-problems", str(EVAL_MAX_PROBLEMS),
        "--batch-size", "2",
    ]
    if NPROC > 1:
        cmd = ["torchrun", "--standalone", f"--nproc_per_node={NPROC}"] + post_eval_args
    else:
        cmd = [sys.executable] + post_eval_args
    print("Running:", " ".join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=WORK_REPO, env=env, check=True)
else:
    print("Skipping comparison eval")


Running: torchrun --standalone --nproc_per_node=2 -m scripts.chat_post_eval -- --models sft=sft:d8_kaggle --models distill=@/kaggle/working/nanochat_cache/chatdistill_checkpoints:d8_kaggle --max-problems 50 --batch-size 2


[W614 03:24:56.435030610 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 03:25:00,416 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 03:25:00,418 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 03:25:00,424 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 03:25:00,426 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 03:25:01.716849759 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 03:25:01.717146613 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

Final: 11/50 (22.00%)
distill | MMLU: 22.00%


2026-06-14 03:28:16,128 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 03:28:16,133 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 03:28:16,144 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 03:28:16,148 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 03:28:16,207 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 03:28:16,221 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k

Rank 1 | 0/25 (0.00%)
Rank 0 | 0/25 (0.00%)
Final: 0/50 (0.00%)
distill | GSM8K: 0.00%


2026-06-14 03:29:47,425 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 03:29:47,429 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 03:29:47,440 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-14 03:29:47,444 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-14 03:29:47,504 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/openai_humaneval.py "HTTP/1.1 404 Not Found"
2026-06-14 03:29:47,504 - httpx - INFO

Rank 1 | 0/25 (0.00%)
Rank 0 | 0/25 (0.00%)
Final: 0/50 (0.00%)
distill | HumanEval: 0.00%
Rank 0 | 10/25 (40.00%)
Rank 1 | 6/25 (24.00%)
Final: 16/50 (32.00%)
distill | SpellingBee: 32.00%

Model   | ARC-Easy | ARC-Challenge | MMLU   | GSM8K | HumanEval | SpellingBee | Mean  
--------+----------+---------------+--------+-------+-----------+-------------+-------
sft     | 30.00%   | 30.00%        | 32.00% | 0.00% | 0.00%     | 96.00%      | 31.33%
distill | 28.00%   | 20.00%        | 22.00% | 0.00% | 0.00%     | 32.00%      | 17.00%

Ranking by mean score:
1. sft: 31.33%
2. distill: 17.00%


## Inspect Saved Comparison Report

In [16]:
report_path = WORK_CACHE / "report" / "chat-post-eval.md"
print(report_path)
print("Exists:", report_path.exists())
if report_path.exists():
    print(report_path.read_text())


/kaggle/working/nanochat_cache/report/chat-post-eval.md
Exists: True
## Chat Post Eval
timestamp: 2026-06-14 03:32:28

- tasks: ['ARC-Easy', 'ARC-Challenge', 'MMLU', 'GSM8K', 'HumanEval', 'SpellingBee']
- num_samples: 1
- temperature: 0.0000
- max_new_tokens: 512
- batch_size: 2
- max_problems: 50
- models: [{'label': 'sft', 'origin': 'sft', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle', 'model_tag': 'd8_kaggle', 'step': 4999, 'resolved_meta_keys': ['model_config', 'step', 'user_config', 'val_bpb']}, {'label': 'distill', 'origin': '/kaggle/working/nanochat_cache/chatdistill_checkpoints', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatdistill_checkpoints/d8_kaggle', 'model_tag': 'd8_kaggle', 'step': 29, 'resolved_meta_keys': ['best_val_loss', 'model_config', 'step', 'user_config']}]
- results: [{'label': 'sft', 'ARC-Easy': 0.3, 'ARC-Challenge': 0.3, 'MMLU': 0.32, 'GSM8K': 0.0, 'HumanEval': 0.0, 'SpellingBee': 0.96}, {'label': 'distill', 'ARC-Eas

## Output Manifest

This records the main distill outputs in `/kaggle/working` for quick inspection.


In [17]:
manifest = {
    "teacher_model": TEACHER_MODEL,
    "total_examples": TOTAL_EXAMPLES,
    "max_new_tokens": MAX_NEW_TOKENS,
    "distill_data": str(DISTILL_DATA),
    "preference_data": str(PREFERENCE_PATH) if PREFERENCE_PATH.exists() else None,
    "distill_checkpoint_dir": str(WORK_CACHE / "chatdistill_checkpoints" / STUDENT_TAG),
    "report": str(WORK_CACHE / "report" / "chat-post-eval.md"),
}
manifest_path = Path("/kaggle/working/nanochat_distill_manifest.json")
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(manifest_path)
print(manifest_path.read_text())

print("Main output files:")
for path in [DISTILL_DATA, PREFERENCE_PATH, manifest_path]:
    if path.exists():
        print(path, path.stat().st_size, "bytes")
for path in sorted((WORK_CACHE / "chatdistill_checkpoints" / STUDENT_TAG).glob("*")):
    print(path, path.stat().st_size, "bytes")


/kaggle/working/nanochat_distill_manifest.json
{
  "teacher_model": "meta-llama/Llama-3.1-8B-Instruct",
  "total_examples": 128,
  "max_new_tokens": 128,
  "distill_data": "/kaggle/working/nanochat_cache/teacher_sft_distill.jsonl",
  "preference_data": null,
  "distill_checkpoint_dir": "/kaggle/working/nanochat_cache/chatdistill_checkpoints/d8_kaggle",
  "report": "/kaggle/working/nanochat_cache/report/chat-post-eval.md"
}
Main output files:
/kaggle/working/nanochat_cache/teacher_sft_distill.jsonl 135099 bytes
/kaggle/working/nanochat_distill_manifest.json 379 bytes
/kaggle/working/nanochat_cache/chatdistill_checkpoints/d8_kaggle/meta_000029.json 827 bytes
/kaggle/working/nanochat_cache/chatdistill_checkpoints/d8_kaggle/model_000029.pt 503342527 bytes


In [18]:
# Optional: save the distill checkpoint cache as a Kaggle Dataset.
import json

DISTILL_MODEL_TAG = STUDENT_TAG
DISTILL_SOURCE_DIR = WORK_CACHE / 'chatdistill_checkpoints' / DISTILL_MODEL_TAG
TOKENIZER_SOURCE_DIR = WORK_CACHE / 'tokenizer'
DISTILL_CACHE_DIR = Path('/kaggle/working/nanochat_d8_distill_cache')

DATASET_ID = 'yshuaiqin/nanochat-d8-distill-cache'
TITLE = 'nanochat d8 distill cache'
VERSION_MESSAGE = f'Save {DISTILL_MODEL_TAG} chat distill checkpoint cache'
UPLOAD_ACTION = 'create'  # use 'version' after the Dataset already exists

assert '/' in DATASET_ID, "DATASET_ID should look like '<username>/<dataset-slug>'."
assert DISTILL_SOURCE_DIR.exists(), f'Missing distill checkpoint directory: {DISTILL_SOURCE_DIR}'
assert TOKENIZER_SOURCE_DIR.exists(), f'Missing tokenizer directory: {TOKENIZER_SOURCE_DIR}'
assert sorted(DISTILL_SOURCE_DIR.glob('model_*.pt')), f'No model_*.pt files found in {DISTILL_SOURCE_DIR}'
assert sorted(DISTILL_SOURCE_DIR.glob('meta_*.json')), f'No meta_*.json files found in {DISTILL_SOURCE_DIR}'

if DISTILL_CACHE_DIR.exists():
    shutil.rmtree(DISTILL_CACHE_DIR)
DISTILL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(WORK_CACHE / 'chatdistill_checkpoints', DISTILL_CACHE_DIR / 'chatdistill_checkpoints')
shutil.copytree(TOKENIZER_SOURCE_DIR, DISTILL_CACHE_DIR / 'tokenizer')

metadata = {
    'title': TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}

metadata_path = DISTILL_CACHE_DIR / 'dataset-metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2))

print('Folder size:')
subprocess.run(['du', '-sh', str(DISTILL_CACHE_DIR)], check=False)

if UPLOAD_ACTION == 'create':
    cmd = [
        'kaggle', 'datasets', 'create',
        '-p', str(DISTILL_CACHE_DIR),
        '--dir-mode', 'tar',
    ]
elif UPLOAD_ACTION == 'version':
    cmd = [
        'kaggle', 'datasets', 'version',
        '-p', str(DISTILL_CACHE_DIR),
        '-m', VERSION_MESSAGE,
        '--dir-mode', 'tar',
    ]
else:
    raise ValueError("UPLOAD_ACTION must be 'create' or 'version'")

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)

if result.returncode != 0:
    raise RuntimeError('Kaggle Dataset upload failed.')

print('Done.')
print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


Folder size:
481M	/kaggle/working/nanochat_d8_distill_cache
Running: kaggle datasets create -p /kaggle/working/nanochat_d8_distill_cache --dir-mode tar
Starting upload for file tokenizer.tar


100%|██████████| 540k/540k [00:00<00:00, 1.20MB/s]


Upload successful: tokenizer.tar (540KB)
Starting upload for file chatdistill_checkpoints.tar


100%|██████████| 480M/480M [00:06<00:00, 73.5MB/s]


Upload successful: chatdistill_checkpoints.tar (480MB)
Dataset creation error: Dataset url's dataset slugs and hashlink are all null
Done.
Dataset URL: https://www.kaggle.com/datasets/yshuaiqin/nanochat-d8-distill-cache


## Serve the Distilled Chat Model

Run this after `chatdistill_checkpoints/d8_kaggle` exists. The first cell starts or restarts the local NanoChat web server from the distilled checkpoint. The second cell provides a small notebook chat loop against that server.


In [ ]:
import time
import requests

DISTILL_CHECKPOINT_DIR = WORK_CACHE / "chatdistill_checkpoints"
DISTILL_MODEL_TAG = STUDENT_TAG
SERVER_PORT = 8000
BASE_URL = f"http://127.0.0.1:{SERVER_PORT}"

model_dir = DISTILL_CHECKPOINT_DIR / DISTILL_MODEL_TAG
assert model_dir.exists(), f"Missing distill checkpoint directory: {model_dir}"
assert sorted(model_dir.glob("model_*.pt")), f"No model_*.pt files found in {model_dir}"
assert sorted(model_dir.glob("meta_*.json")), f"No meta_*.json files found in {model_dir}"

try:
    if server.poll() is None:
        print("Stopping existing NanoChat server...")
        server.terminate()
        server.wait(timeout=10)
        print("Stopped existing server.")
except NameError:
    pass
except Exception as exc:
    print("Could not stop existing server cleanly:", exc)
    try:
        server.kill()
        server.wait(timeout=10)
        print("Killed existing server.")
    except Exception:
        pass

messages = []

server_env = env.copy()
server_env["NANOCHAT_DISABLE_COMPILE"] = "1"
server_env["TORCH_COMPILE_DISABLE"] = "1"
server_env["OMP_NUM_THREADS"] = "1"

cmd = [
    sys.executable,
    "-m", "scripts.chat_web",
    "--checkpoint-dir", str(DISTILL_CHECKPOINT_DIR),
    "--model-tag", DISTILL_MODEL_TAG,
    "--num-gpus", "1",
    "--port", str(SERVER_PORT),
]

print("Starting NanoChat server:")
print(" ".join(cmd))
server = subprocess.Popen(cmd, cwd=WORK_REPO, env=server_env)
print(f"Server process started with PID {server.pid}.")

SERVER_READY = False
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError(f"NanoChat server exited early with code {server.returncode}")
    try:
        response = requests.get(f"{BASE_URL}/health", timeout=2)
        if response.ok:
            SERVER_READY = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if SERVER_READY:
    print(f"NanoChat server is ready: {BASE_URL}")
else:
    print(f"NanoChat server is still loading. Wait a bit, then use: {BASE_URL}")

In [ ]:
import json
import requests

BASE = globals().get("BASE_URL", "http://127.0.0.1:8000")
messages = []

def ask(prompt, temperature=0.8, top_k=50, max_tokens=512):
    messages.append({"role": "user", "content": prompt})

    payload = {
        "messages": messages,
        "temperature": temperature,
        "top_k": top_k,
        "max_tokens": max_tokens,
    }

    print("Assistant:", end=" ", flush=True)
    answer = ""

    with requests.post(f"{BASE}/chat/completions", json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data: "):
                continue

            data = json.loads(line[len("data: "):])
            if data.get("done"):
                break

            token = data.get("token", "")
            answer += token
            print(token, end="", flush=True)

    print()
    messages.append({"role": "assistant", "content": answer})
    return answer

print(f"Chatting with {BASE}. Type q, quit, or exit to stop. Type reset to clear history.")
while True:
    prompt = input("\nYou: ")
    command = prompt.strip().lower()
    if command in {"q", "quit", "exit"}:
        break
    if command == "reset":
        messages.clear()
        print("Chat history cleared.")
        continue
    ask(prompt)

In [ ]:
# Stop the NanoChat web server started by the serving cell.
import os
import time

SERVER_PORT = globals().get('SERVER_PORT', 8000)
stopped_any = False

server_process = globals().get('server')
if server_process is not None:
    if server_process.poll() is None:
        print(f'Stopping NanoChat server process {server_process.pid}...')
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print('NanoChat server stopped.')
        except subprocess.TimeoutExpired:
            print('Server did not stop after terminate(); killing it...')
            server_process.kill()
            server_process.wait(timeout=10)
            print('NanoChat server killed.')
        stopped_any = True
    else:
        print(f'NanoChat server process already exited with code {server_process.returncode}.')
else:
    print('No notebook `server` variable found.')

try:
    import psutil

    current_pid = os.getpid()
    fallback_processes = []
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        if proc.info['pid'] == current_pid:
            continue
        cmdline = ' '.join(proc.info.get('cmdline') or [])
        if 'scripts.chat_web' not in cmdline:
            continue
        try:
            listening_on_port = any(
                conn.status == psutil.CONN_LISTEN
                and conn.laddr
                and conn.laddr.port == SERVER_PORT
                for conn in proc.net_connections(kind='inet')
            )
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            listening_on_port = False
        if listening_on_port:
            fallback_processes.append(proc)

    for proc in fallback_processes:
        print(f'Stopping fallback chat_web process {proc.pid} on port {SERVER_PORT}...')
        proc.terminate()
    gone, alive = psutil.wait_procs(fallback_processes, timeout=10)
    for proc in alive:
        print(f'Killing fallback chat_web process {proc.pid}...')
        proc.kill()
    if fallback_processes:
        stopped_any = True
except Exception as exc:
    print('Fallback process scan skipped:', exc)

if stopped_any:
    time.sleep(2)
    print('Server shutdown complete.')
else:
    print('No running NanoChat server process found.')